# Factor selection workbook

Here I am going to be parsing the data to figure out which factors to select for the multi factor model. These selections will be based on the code below and effectivness of these factors

## Imports

In [2]:
import pandas as pd
import duckdb

## Initializing all the path

In [3]:
db_path = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\fintech_factors.duckdb"
parquet_file = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\CRSPdF20022025.parquet"

## Creating the new DuckDB database

In [4]:
con = duckdb.connect(db_path)

con.execute(f"""
CREATE TABLE IF NOT EXISTS fintech_data AS SELECT * FROM read_parquet ('{parquet_file}')
""")

con.execute("SELECT * FROM fintech_data LIMIT 5").fetch_df()

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.10,-0.030568,-0.030568,6603.0,11.35,11.49,11.19,11.1000
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.60,-0.029323,-0.029323,55000.0,9.89,9.60,9.27,9.6000
2,23326910,2002-01-02,10012,7969,1.0,1.0,20924.0,3.25,0.041667,0.041667,50975.0,3.25,3.29,3.24,3.0600
3,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.05,-0.160000,-0.160000,53250.0,1.05,1.30,1.03,1.0497
4,00103110,2002-01-02,10025,7975,1.0,1.0,7794.0,23.80,-0.006678,-0.006678,7030.0,23.90,24.00,23.80,23.4000


## Data Cleanning

In [10]:
con.execute(f"""
    CREATE OR REPLACE TABLE crsp_clean AS 
            SELECT
            cusip,
            CAST(date as DATE) AS trade_date,
            permno,
            permco,
            cfacpr,
            cfacshr,
            CASE WHEN shrout <= 0 THEN NULL ELSE shrout END AS shrout,
            CASE WHEN prc <= 0 THEN NULL ELSE prc END AS prc,
            ret,
            retx,
            CASE WHEN vol <= 0 THEN NULL ELSE vol END AS vol,
            CASE WHEN ask <= 0 THEN NULL ELSE ask END AS ask,
            askhi,
            CASE WHEN bid <= 0 THEN NULL ELSE bid END AS bid,
            bidlo
        FROM fintech_data;
""" 
)

con.execute("SHOW TABLES;").fetch_df()

,name
0,crsp_clean
1,fintech_data


## Forming some factors

In [12]:
con.execute(f"""
        CREATE OR REPLACE TABLE factors_daily AS 
        SELECT *,
        CASE
            WHEN prc IS NULL OR shrout IS NULL THEN NULL
            ELSE prc * shrout * 100.0
        END AS mkt_cap,
        CASE
            WHEN prc IS NULL OR shrout IS NULL OR (prc * shrout * 100) < 0 THEN NULL
            ELSE ln(prc * shrout * 100.0)
        END AS ln_mkt_cap,
        CASE 
            WHEN vol IS NULL OR shrout IS NULL OR shrout = 0  THEN NULL
            ELSE vol / shrout
        END AS turnover,
        CASE 
            WHEN ask IS NULL OR bid IS NULL OR (ask + bid) = 0 THEN NULL
            ELSE (ask - bid) / ((ask + bid) / 2.0)
        END AS bid_ask_spread
        FROM crsp_clean;      
""")
con.execute("SHOW TABLES;").fetch_df()

,name
0,crsp_clean
1,factors_daily
2,fintech_data


In [ ]:
con.execute